# MoMo-FDVS P12 — governed CNN training

This is the only authorised reportable training surface for P12. It uses the frozen P10 source-group splits, training-only augmentation, validation-only threshold selection and one final held-out evaluation. The dataset contains only generic controlled examples, so no provider-wide or production claim is allowed.

In [ ]:
from pathlib import Path
import shutil
import subprocess
import sys

REPOSITORY_URL = "https://github.com/davidagyekum/momo-fraud-detection.git"
TRAINING_COMMIT_SHA = "02d8967136853c5c46eaa0babe44a7327c843a32"
WORKSPACE = Path("/content/momo-fraud-detection")
assert sys.version_info[:2] == (3, 12), sys.version
if WORKSPACE.exists():
    shutil.rmtree(WORKSPACE)
subprocess.run(["git", "clone", "--quiet", REPOSITORY_URL, str(WORKSPACE)], check=True)
subprocess.run(["git", "checkout", "--quiet", TRAINING_COMMIT_SHA], cwd=WORKSPACE, check=True)
head = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=WORKSPACE, text=True).strip()
assert head == TRAINING_COMMIT_SHA, (head, TRAINING_COMMIT_SHA)
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--requirement", "ml/requirements-dev.lock"], cwd=WORKSPACE, check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--requirement", "ml/requirements-training.lock"], cwd=WORKSPACE, check=True)
print({"training_commit_sha": head, "python": sys.version})

In [ ]:
import json
import os

environment = os.environ.copy()
environment["PYTHONPATH"] = str(WORKSPACE / "ml" / "src")
subprocess.run([sys.executable, "scripts/verify_ml.py"], cwd=WORKSPACE, env=environment, check=True)
dataset_report = json.loads((WORKSPACE / "ml/data/controlled/image_dataset_report.json").read_text())
assert dataset_report["training_executed"] is False
assert dataset_report["group_intersections"] == {"train_test": [], "train_validation": [], "validation_test": []}
assert dataset_report["preprocessing_schema_hash"] == "8510a396d3115887f8ebff88414f75f9ea5b353f375d93cfdf65f488d55df616"
print("P12 Colab preflight passed; no model training has executed.")

## STOP — owner approval required

Do not run the next cell until the project owner has reviewed the preflight output and explicitly approved the controlled-only CNN training run. The next cell fits model weights.

In [ ]:
OUTPUT_DIR = Path("/content/p12-image-output")
if OUTPUT_DIR.exists():
    shutil.rmtree(OUTPUT_DIR)
command = [
    sys.executable, "-m", "momo_fdvs_ml", "train-image",
    "--manifest", str(WORKSPACE / "ml/data/controlled/manifest.csv"),
    "--root", str(WORKSPACE / "ml/data/controlled"),
    "--output-dir", str(OUTPUT_DIR),
    "--model-version", "image-mobilenetv3-controlled-v1",
    "--training-commit-sha", TRAINING_COMMIT_SHA,
]
subprocess.run(command, cwd=WORKSPACE, env=environment, check=True)

In [ ]:
import hashlib

report = json.loads((OUTPUT_DIR / "image_evaluation_report.json").read_text())
assert report["training_commit_sha"] == TRAINING_COMMIT_SHA
assert report["dataset_scope"] == "controlled_synthetic_only"
assert report["dataset_manifest_hash"] == "51d12132904f461fb4bec6a5d0eda9cff5dd94961a48129b7dd75359b38ead1f"
assert report["split_hash"] == "08008637eb661634eb93fee4d4ac74d82da598b2b0ff28f188f9641e47e933f9"
assert report["preprocessing_schema_hash"] == "8510a396d3115887f8ebff88414f75f9ea5b353f375d93cfdf65f488d55df616"
assert report["held_out_test"]["source_group_count"] == 1
assert report["held_out_test"]["sample_count"] == 2
artifact = OUTPUT_DIR / "image-mobilenetv3-controlled-v1.keras"
artifact_sha = hashlib.sha256(artifact.read_bytes()).hexdigest()
assert artifact_sha == report["artifact"]["sha256"]
print(json.dumps({"artifact_sha256": artifact_sha, "held_out_test": report["held_out_test"], "cpu_inference": report["cpu_inference"], "acceptance_passed": report["acceptance_passed"], "limitations": report["limitations"]}, indent=2, sort_keys=True))

In [ ]:
subprocess.run([sys.executable, "-m", "momo_fdvs_ml", "verify-image-artifact", "--artifact", str(artifact), "--sha256", artifact_sha, "--schema-hash", report["preprocessing_schema_hash"]], cwd=WORKSPACE, env=environment, check=True)
print("Independent Keras artifact hash/schema/shape verification passed.")

In [ ]:
import zipfile
from google.colab import files

output_package = Path("/content/P12_COLAB_OUTPUTS.zip")
with zipfile.ZipFile(output_package, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for filename in ["image_evaluation_report.json", "IMAGE_MODEL_CARD.md", "image_registry_payload.json", "image_confusion_matrix.png"]:
        archive.write(OUTPUT_DIR / filename, arcname=f"safe-evidence/{filename}")
    archive.write(artifact, arcname=f"private-artifact/{artifact.name}")
print({"output_package": str(output_package), "private_artifact_sha256": artifact_sha, "warning": "Never commit private-artifact/ to Git."})
files.download(str(output_package))